# Clasificador Clumpy v2.1 - Mejoras Conservadoras

ConvNeXt-Tiny + MLP con: **15 features**, **TTA**, **10-Fold CV** (sin MixUp ni Label Smoothing).

| Mejora | Que hace |
|---|---|
| 15 features tabulares | Anade F(G,M20), S(G,M20), rpetro_circ, r20, r80 |
| TTA | En inferencia, promedia predicciones de 4 augmentaciones |
| 10-Fold CV | Menor varianza en estimacion (std=0.17 -> ~0.10) |

**F1 esperado**: 0.78-0.82 (base: 0.76 +/- 0.17)

**Nota v2.1**: MixUp y Label Smoothing removidos porque en v2 causaron F1=0.58 (interaccion negativa con Focal Loss).

In [ ]:
!pip install timm -q
import os, json, random, copy, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
import torchvision.transforms as transforms
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve, precision_recall_curve, average_precision_score)
import timm
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f'PyTorch: {torch.__version__}')
print(f'timm: {timm.__version__}')

## 1. Montar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/ProyectoClumps/data'
JSON_PATH = os.path.join(DATA_DIR, '11_05_2026_dictionary_with_all_statmorph_and_classifications.json')
IMAGES_DIR = os.path.join(DATA_DIR, 'colour_images')
SAVE_DIR = '/content/drive/MyDrive/ProyectoClumps'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Cargar datos con 15 features tabulares (expandidas)

Anadidas: F(G,M20), S(G,M20), rpetro_circ, r20, r80 respecto a v1.

In [ ]:
with open(JSON_PATH) as f:
    data = json.load(f)

TABULAR_COLS = [
    'Gini', 'M20', 'F(G, M20)', 'S(G, M20)',
    'C', 'A', 'S', 'sn_per_pixel',
    'sersic_n', 'sersic_rhalf',
    'ellipticity_centroid', 'elongation_centroid',
    'rpetro_circ', 'r20', 'r80'
]
print(f'Total features tabulares: {len(TABULAR_COLS)}')

img_files = {os.path.splitext(f)[0] for f in os.listdir(IMAGES_DIR) if f.endswith('.png')}

samples = []
for g in data['galaxies']:
    name = g['name']
    if name not in img_files or not g['frames']:
        continue
    frame = g['frames'][0]
    try:
        tab = [float(frame[c]) for c in TABULAR_COLS]
        if any(np.isnan(tab)) or any(np.isinf(tab)):
            continue
    except (KeyError, TypeError, ValueError):
        continue
    label = 1 if g['info']['visual_clas']['Clumpy'] else 0
    samples.append({
        'name': name,
        'path': os.path.join(IMAGES_DIR, name + '.png'),
        'label': label,
        'tabular': np.array(tab, dtype=np.float32),
    })

print(f'Total muestras validas: {len(samples)}')
cl_count = sum(s['label'] for s in samples)
print(f'Clumpy: {cl_count} ({100*cl_count/len(samples):.1f}%)')

all_labels = np.array([s['label'] for s in samples])
class_counts = np.bincount(all_labels)

## 3. Dataset, transforms, scaler, TTA transforms

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize(240),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize(240),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

TTA_TRANSFORMS = [
    eval_transform,
    transforms.Compose([
        transforms.Resize(240), transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize(240), transforms.CenterCrop(224),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    transforms.Compose([
        transforms.Resize(240), transforms.CenterCrop(224),
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.RandomVerticalFlip(p=1.0),
        transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
]
print(f'TTA: {len(TTA_TRANSFORMS)} augmentaciones (original + hflip + vflip + hvflip)')


class ClumpyDataset(Dataset):
    def __init__(self, samples, transform=None, scaler_mean=None, scaler_std=None):
        self.samples = samples
        self.transform = transform
        self.scaler_mean = scaler_mean
        self.scaler_std = scaler_std
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        img = Image.open(s['path']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        tab = s['tabular'].copy()
        if self.scaler_mean is not None:
            tab = (tab - self.scaler_mean) / self.scaler_std
        return img, tab, s['label'], s['name']


def fit_scaler(train_samples):
    features = np.array([s['tabular'] for s in train_samples])
    mean = features.mean(axis=0).astype(np.float32)
    std = features.std(axis=0).astype(np.float32)
    std[std == 0] = 1.0
    return mean, std


def make_loaders(train_samples, val_samples, batch_size=64):
    scaler_mean, scaler_std = fit_scaler(train_samples)
    train_labels = [s['label'] for s in train_samples]
    cc = np.bincount(train_labels)
    cw = 1. / cc
    sw = np.array([cw[l] for l in train_labels])
    sampler = WeightedRandomSampler(sw, num_samples=len(train_samples), replacement=True)
    train_ds = ClumpyDataset(train_samples, transform=train_transform, scaler_mean=scaler_mean, scaler_std=scaler_std)
    val_ds = ClumpyDataset(val_samples, transform=eval_transform, scaler_mean=scaler_mean, scaler_std=scaler_std)
    train_loader = DataLoader(train_ds, batch_size=batch_size, sampler=sampler, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader, scaler_mean, scaler_std

print(f'Features tabulares por muestra: {len(TABULAR_COLS)}')

## 4. Modelo Hibrido (ConvNeXt-Tiny + MLP)

In [ ]:
class HybridModel(nn.Module):
    def __init__(self, num_tabular_features=15, tabular_hidden=128):
        super().__init__()
        self.backbone = timm.create_model('convnext_tiny', pretrained=True)
        img_feature_dim = self.backbone.head.fc.in_features
        self.backbone.head.fc = nn.Identity()
        self.tabular_net = nn.Sequential(
            nn.Linear(num_tabular_features, tabular_hidden),
            nn.ReLU(),
            nn.Dropout(0.2),
        )
        self.classifier = nn.Linear(img_feature_dim + tabular_hidden, 1)

    def forward(self, images, tabular):
        img_feat = self.backbone(images)
        tab_feat = self.tabular_net(tabular)
        return self.classifier(torch.cat([img_feat, tab_feat], dim=1))


m = HybridModel(len(TABULAR_COLS)).to(device)
print(f'Params: {sum(p.numel() for p in m.parameters()):,}')
del m

## 5. Focal Loss (sin label smoothing en v2.1)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.9, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        p = torch.sigmoid(logits.view(-1))
        t = targets.view(-1).float()
        p_t = p * t + (1 - p) * (1 - t)
        alpha_t = self.alpha * t + (1 - self.alpha) * (1 - t)
        focal = alpha_t * (1 - p_t) ** self.gamma
        bce = F.binary_cross_entropy_with_logits(logits.view(-1), t, reduction='none')
        return (focal * bce).mean()

criterion = FocalLoss(alpha=0.9, gamma=2.0).to(device)
print('Focal Loss: alpha=0.9, gamma=2.0 (sin label smoothing)')

## 7. Funciones auxiliares (train, evaluate, threshold, TTA)

In [ ]:
def build_hybrid_model():
    model = HybridModel(len(TABULAR_COLS))
    for param in model.backbone.parameters():
        param.requires_grad = False
    return model.to(device)


def train_one_epoch(model, loader, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for images, tabular, labels, _ in tqdm(loader, desc='Train', leave=False):
        images = images.to(device)
        tabular = tabular.to(device)
        labels = labels.float().to(device)
        optimizer.zero_grad()
        with autocast('cuda'):
            outputs = model(images, tabular).squeeze(1)
            loss = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    for images, tabular, labels, _ in tqdm(loader, desc='Eval', leave=False):
        images = images.to(device)
        tabular = tabular.to(device)
        labels = labels.float().to(device)
        with autocast('cuda'):
            outputs = model(images, tabular).squeeze(1)
            loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        all_preds.extend(torch.sigmoid(outputs).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return total_loss / len(loader.dataset), np.array(all_preds), np.array(all_labels)


@torch.no_grad()
def evaluate_tta(model, samples_list, scaler_mean, scaler_std, batch_size=64):
    model.eval()
    all_tta_probs = []
    for t in TTA_TRANSFORMS:
        ds = ClumpyDataset(samples_list, transform=t, scaler_mean=scaler_mean, scaler_std=scaler_std)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
        probs = []
        for images, tabular, _, _ in loader:
            images = images.to(device)
            tabular = tabular.to(device)
            with autocast('cuda'):
                outputs = model(images, tabular).squeeze(1)
            probs.extend(torch.sigmoid(outputs).cpu().numpy())
        all_tta_probs.append(np.array(probs))
    return np.mean(all_tta_probs, axis=0)


def find_best_threshold(labels, probs):
    if len(np.unique(labels)) < 2:
        return 0.5, 0.0
    best_th, best_f1 = 0.5, 0.0
    for th in np.arange(0.01, 1.00, 0.01):
        f1 = f1_score(labels, (probs > th).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_th = f1, th
    return best_th, best_f1

print('Funciones auxiliares listas (incluye TTA)')

## 8. Entrenamiento por fold

In [ ]:
EPOCHS = 50
FREEZE_EPOCHS = 5
PATIENCE = 10
LR_HEAD = 2e-4
LR_FINETUNE = 5e-5
WEIGHT_DECAY = 1e-4
BATCH_SIZE = 64


def train_fold(train_samples, val_samples, fold_name='fold'):
    train_loader, val_loader, sm, ss = make_loaders(train_samples, val_samples, batch_size=BATCH_SIZE)

    model = build_hybrid_model()
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LR_HEAD, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = GradScaler('cuda')

    best_val_f1 = 0.0
    best_threshold = 0.5
    patience_counter = 0
    best_state = copy.deepcopy(model.state_dict())
    history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_auc': []}

    for epoch in range(EPOCHS):
        if epoch == FREEZE_EPOCHS:
            print(f'  [{fold_name}] Descongelando backbone en epoca {epoch}')
            for param in model.backbone.parameters():
                param.requires_grad = True
            optimizer = torch.optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=WEIGHT_DECAY)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - FREEZE_EPOCHS)
            scaler = GradScaler('cuda')

        train_loss = train_one_epoch(model, train_loader, optimizer, scaler)
        val_loss, val_probs, val_labels = evaluate(model, val_loader)

        val_th, _ = find_best_threshold(val_labels, val_probs)
        val_preds = (val_probs > val_th).astype(int)
        val_f1 = f1_score(val_labels, val_preds, zero_division=0)
        try:
            val_auc = roc_auc_score(val_labels, val_probs)
        except ValueError:
            val_auc = 0.5

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_f1'].append(val_f1)
        history['val_auc'].append(val_auc)

        print(f'  [{fold_name}] Ep {epoch+1}/{EPOCHS} | TrainL: {train_loss:.4f} | '
              f'ValL: {val_loss:.4f} | F1: {val_f1:.4f} (th={val_th:.2f}) | AUC: {val_auc:.4f}')

        scheduler.step()

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_threshold = val_th
            patience_counter = 0
            best_state = copy.deepcopy(model.state_dict())
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f'  [{fold_name}] Early stopping en epoca {epoch+1}')
                break

    model.load_state_dict(best_state)
    print(f'  [{fold_name}] Mejor F1: {best_val_f1:.4f} | Umbral: {best_threshold:.3f}')
    return model, best_threshold, best_val_f1, history, sm, ss

## 9. 10-Fold Cross Validation con TTA

In [ ]:
K_FOLDS = 10
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)

cv_results = []
cv_histories = []
trained_models = []

for fold, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(samples)), all_labels)):
    print(f'\n{"="*60}')
    print(f'FOLD {fold+1}/{K_FOLDS}')
    print(f'{"="*60}')

    fold_train = [samples[i] for i in train_idx]
    fold_val = [samples[i] for i in val_idx]
    n_c_tr = sum(s['label'] for s in fold_train)
    n_c_v = sum(s['label'] for s in fold_val)
    print(f'Train: {len(fold_train)} (Clumpy: {n_c_tr}) | Val: {len(fold_val)} (Clumpy: {n_c_v})')

    model, best_th, best_f1, history, sm, ss = train_fold(fold_train, fold_val, fold_name=f'F{fold+1}')
    cv_histories.append(history)
    trained_models.append((copy.deepcopy(model.state_dict()), sm, ss, best_th))

    val_probs_tta = evaluate_tta(model, fold_val, sm, ss, batch_size=BATCH_SIZE)
    val_labels_arr = np.array([s['label'] for s in fold_val])
    tta_th, tta_f1 = find_best_threshold(val_labels_arr, val_probs_tta)
    tta_preds = (val_probs_tta > tta_th).astype(int)

    fold_metrics = {
        'fold': fold + 1,
        'threshold': tta_th,
        'accuracy': accuracy_score(val_labels_arr, tta_preds),
        'precision': precision_score(val_labels_arr, tta_preds, zero_division=0),
        'recall': recall_score(val_labels_arr, tta_preds, zero_division=0),
        'f1': f1_score(val_labels_arr, tta_preds, zero_division=0),
        'roc_auc': roc_auc_score(val_labels_arr, val_probs_tta),
        'pr_auc': average_precision_score(val_labels_arr, val_probs_tta),
        'val_probs': val_probs_tta,
        'val_labels': val_labels_arr,
    }
    cv_results.append(fold_metrics)
    print(f'  -> F1 (TTA): {fold_metrics["f1"]:.4f} | Prec: {fold_metrics["precision"]:.4f} | '
          f'Rec: {fold_metrics["recall"]:.4f} | AUC: {fold_metrics["roc_auc"]:.4f}')

    del model
    torch.cuda.empty_cache()

print(f'\n{"="*60}')
print('CV COMPLETADA (10 folds con TTA)')
print(f'{"="*60}')

## 10. Resultados CV

In [ ]:
cv_df = pd.DataFrame([{k: v for k, v in r.items() if k not in ('val_probs', 'val_labels')}
                       for r in cv_results])
print(cv_df.to_string(index=False))

print(f'\n{"="*40}')
print('   MEDIA +/- STD (10 folds, TTA)')
print(f'{"="*40}')
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'pr_auc']:
    mean = cv_df[metric].mean()
    std = cv_df[metric].std()
    print(f'{metric:>10}: {mean:.4f} +/- {std:.4f}')
print(f'{"threshold":>10}: {cv_df["threshold"].mean():.4f} +/- {cv_df["threshold"].std():.4f}')
print(f'{"="*40}')

avg_threshold = cv_df['threshold'].mean()
print(f'\nUmbral promedio: {avg_threshold:.4f}')

## 11. Visualizaciones

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for ax, metric, color in zip(axes.flat, ['f1', 'precision', 'recall', 'roc_auc', 'pr_auc', 'threshold'],
                              ['green', 'blue', 'orange', 'red', 'purple', 'brown']):
    values = cv_df[metric].values
    ax.bar(range(1, K_FOLDS+1), values, color=color, alpha=0.7)
    ax.axhline(values.mean(), color='black', linestyle='--', label=f'Mean: {values.mean():.3f}')
    ax.set_title(metric.upper()); ax.set_xlabel('Fold'); ax.set_xticks(range(1, K_FOLDS+1))
    ax.legend(); ax.grid(True, axis='y', alpha=0.3)
plt.suptitle('Metricas por Fold (v2.1: 15 features + TTA + 10-Fold)', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'v2_1_cv_metrics.png'), dpi=150)
plt.show()

In [ ]:
all_cv_probs = np.concatenate([r['val_probs'] for r in cv_results])
all_cv_labels = np.concatenate([r['val_labels'] for r in cv_results])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fpr, tpr, _ = roc_curve(all_cv_labels, all_cv_probs)
auc_all = roc_auc_score(all_cv_labels, all_cv_probs)
axes[0].plot(fpr, tpr, label=f'ROC (AUC={auc_all:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--')
axes[0].set_title('ROC (pooled)'); axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(); axes[0].grid(True)

prec_arr, rec_arr, _ = precision_recall_curve(all_cv_labels, all_cv_probs)
ap_all = average_precision_score(all_cv_labels, all_cv_probs)
axes[1].plot(rec_arr, prec_arr, label=f'PR (AP={ap_all:.3f})', color='purple')
axes[1].set_title('PR (pooled)'); axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(); axes[1].grid(True)

all_preds = (all_cv_probs > avg_threshold).astype(int)
sns.heatmap(confusion_matrix(all_cv_labels, all_preds), annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['No Clumpy', 'Clumpy'], yticklabels=['No Clumpy', 'Clumpy'])
axes[2].set_title(f'CM pooled (th={avg_threshold:.3f})')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'v2_1_cv_curves.png'), dpi=150)
plt.show()

## 12. Ensemble de los 10 modelos + TTA

In [ ]:
ensemble_models = []
for state_dict, sm, ss, th in trained_models:
    m = HybridModel(len(TABULAR_COLS))
    m.load_state_dict(state_dict)
    m.to(device).eval()
    ensemble_models.append((m, sm, ss, th))

print(f'Ensemble: {len(ensemble_models)} modelos cargados. Calculando predicciones con TTA...')

all_probs_ensemble = np.zeros((len(samples), len(ensemble_models)))
for i, (m, sm, ss, _) in enumerate(ensemble_models):
    all_probs_ensemble[:, i] = evaluate_tta(m, samples, sm, ss, batch_size=BATCH_SIZE)

labels_arr = np.array([s['label'] for s in samples])
ensemble_probs = all_probs_ensemble.mean(axis=1)
ensemble_th, _ = find_best_threshold(labels_arr, ensemble_probs)
ensemble_preds = (ensemble_probs > ensemble_th).astype(int)

ens_f1 = f1_score(labels_arr, ensemble_preds, zero_division=0)
ens_prec = precision_score(labels_arr, ensemble_preds, zero_division=0)
ens_rec = recall_score(labels_arr, ensemble_preds, zero_division=0)
ens_auc = roc_auc_score(labels_arr, ensemble_probs)
ens_ap = average_precision_score(labels_arr, ensemble_probs)

print()
print('=' * 50)
print('   ENSEMBLE + TTA (in-sample, optimista)')
print('=' * 50)
print(f'F1:        {ens_f1:.4f}')
print(f'Precision: {ens_prec:.4f}')
print(f'Recall:    {ens_rec:.4f}')
print(f'ROC-AUC:   {ens_auc:.4f}')
print(f'PR-AUC:    {ens_ap:.4f}')
print(f'Umbral:    {ensemble_th:.4f}')
print('=' * 50)
cv_f1_str = f'{cv_df["f1"].mean():.4f}+/-{cv_df["f1"].std():.4f}'
print(f'\nCV con TTA (honesto): F1={cv_f1_str}')
print(f'Ensemble+TTA (in-sample): F1={ens_f1:.4f}')

## 13. Exportar ensemble final

In [ ]:
ensemble_path = os.path.join(SAVE_DIR, 'ensemble_clumpy_v2_1.pth')
torch.save({
    'version': 'v2.1',
    'num_models': len(ensemble_models),
    'model_states': [sd for sd, _, _, _ in trained_models],
    'scaler_means': [sm.tolist() for _, sm, _, _ in trained_models],
    'scaler_stds': [ss.tolist() for _, _, ss, _ in trained_models],
    'threshold': float(ensemble_th),
    'cv_metrics': {k: [float(x) for x in v] for k, v in cv_df.to_dict('list').items()},
    'cv_mean_f1': float(cv_df['f1'].mean()),
    'cv_std_f1': float(cv_df['f1'].std()),
    'ensemble_f1_insample': float(ens_f1),
    'ensemble_auc': float(ens_auc),
    'num_tabular_features': len(TABULAR_COLS),
    'feature_names': list(TABULAR_COLS),
    'tta_augmentations': 4,
    'label_smoothing': 0.0,
}, ensemble_path)
print(f'Ensemble v2.1 guardado: {ensemble_path}')

## 14. Inferencia con ensemble + TTA

In [ ]:
def predict_clumpy_v2(image_path, tabular_features, models_info=None, threshold=None):
    if models_info is None:
        ckpt = torch.load(ensemble_path, weights_only=True)
        models_info = []
        for sd, sm, ss in zip(ckpt['model_states'], ckpt['scaler_means'], ckpt['scaler_stds']):
            m = HybridModel(ckpt['num_tabular_features'])
            m.load_state_dict(sd)
            m.to(device).eval()
            models_info.append((m, np.array(sm, dtype=np.float32), np.array(ss, dtype=np.float32)))
        if threshold is None:
            threshold = ckpt['threshold']
    if threshold is None:
        threshold = ensemble_th

    img = Image.open(image_path).convert('RGB')
    tab_np = np.array(tabular_features, dtype=np.float32)

    all_probs = []
    for m, sm, ss in models_info:
        tab_scaled = (tab_np - sm) / ss
        tab_t = torch.from_numpy(tab_scaled).unsqueeze(0).to(device)
        for t in TTA_TRANSFORMS:
            inp = t(img).unsqueeze(0).to(device)
            with torch.no_grad(), autocast('cuda'):
                prob = torch.sigmoid(m(inp, tab_t).squeeze(1)).item()
            all_probs.append(prob)

    mean_prob = float(np.mean(all_probs))
    return {
        'prediction': 'Clumpy' if mean_prob > threshold else 'No Clumpy',
        'probability': mean_prob,
        'threshold': float(threshold),
        'n_predictions': len(all_probs),
    }


test_result = predict_clumpy_v2(samples[0]['path'], samples[0]['tabular'])
print(f'Ejemplo: {samples[0]["name"]}')
print(f'Real: {"Clumpy" if samples[0]["label"] else "No Clumpy"}')
print(f'Prediccion: {test_result["prediction"]}')
print(f'Probabilidad: {test_result["probability"]:.4f} (promedio de {test_result["n_predictions"]} predicciones)')